# 图表复制工作流 重构

本项目是一个图表数据提取与处理工作流，主要用于从图表图像中提取数据点，计算商品指数与金融资产的比率，并将结果写入数据库。

## 1. 项目概述

### 1.1 功能描述

本工作流包含两大核心功能：

1. **图表数据提取**：通过OpenCV图像处理技术，从图表图片中识别曲线并提取数据点
   - 支持交互式选择数据区域和样本点
   - 支持多种日期格式（YYYY/MM, YYYY-MM-DD）
   - 支持不同频率的数据点（周度、月度、年度）

2. **商品比率计算**：从EDB数据库获取指标数据，计算CRB商品指数与金融资产比率
   - 计算公式：`CRB CMDT Index / (0.5 * (FLOT US Equity + .EARNREV G Index))`
   - 基准日期：2020-11-12
   - 输出周度数据

### 1.2 数据源

- **数据库**：EDB数据库 (edb.edbdata表)
- **指标数据**：
  - CRB CMDT Index：CRB商品指数
  - FLOT US Equity：美国股票指数
  - .EARNREV G Index：盈利修正指数
- **图表图像**：支持URL或本地文件路径

### 1.3 输出结果

- **商品比率数据**：写入`全球实物资产比金融资产`表
- **图像提取数据**：
  - 控制台输出检测点列表
  - 保存标注图像到`detected_points.png`
  - 缓存样本点到`sample_points_cache.json`

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import os
import sys
import json
import logging
from datetime import datetime, timedelta

# 第三方库
import cv2
import numpy as np
import pandas as pd
import requests
import sqlalchemy
from sqlalchemy import create_engine
from dateutil.relativedelta import relativedelta

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

print("依赖导入完成")

### 2.2 配置参数

In [ ]:
# 从config模块导入配置
from config import (
    DB_HOST, DB_PORT, DB_USER, DB_PASSWORD, DB_NAME,
    BASE_DATE, TARGET_BASE_VALUE, TRADE_CODES,
    get_database_engine
)

# 显示配置信息
print(f"数据库主机: {DB_HOST}")
print(f"数据库端口: {DB_PORT}")
print(f"数据库名称: {DB_NAME}")
print(f"基准日期: {BASE_DATE}")
print(f"目标基准值: {TARGET_BASE_VALUE}")
print(f"交易代码: {TRADE_CODES}")

## 3. 数据获取

### 3.1 数据源连接

In [ ]:
# 创建数据库引擎
engine = get_database_engine()
print("数据库引擎创建成功")

# 测试连接
try:
    with engine.connect() as conn:
        result = conn.execute(sqlalchemy.text("SELECT 1"))
        print("数据库连接测试成功")
except Exception as e:
    print(f"数据库连接失败: {e}")

### 3.2 数据读取

In [ ]:
def get_data_from_database(engine, trade_codes, start_date):
    """
    从数据库查询所需数据
    
    参数:
        engine: SQLAlchemy引擎
        trade_codes: 交易代码列表
        start_date: 起始日期
        
    返回:
        dict: {trade_code: DataFrame} 字典
    """
    logger.info(f"开始从数据库查询数据，起始日期: {start_date}")
    
    data_frames = {}
    
    for trade_code in trade_codes:
        sql = f"""
        SELECT DT, CLOSE, TRADE_CODE
        FROM edb.edbdata
        WHERE TRADE_CODE = %s
        AND DT >= %s
        ORDER BY DT
        """
        
        df = pd.read_sql(sql, engine, params=(trade_code, start_date))
        df['DT'] = pd.to_datetime(df['DT'])
        df = df.set_index('DT')
        data_frames[trade_code] = df
        logger.info(f"查询到 {trade_code} 数据 {len(df)} 条")
    
    return data_frames

# 读取数据
data_frames = get_data_from_database(engine, TRADE_CODES, BASE_DATE)
print(f"\n共读取 {len(data_frames)} 个指标的数据")

## 4. 数据处理

### 4.1 数据清洗

In [ ]:
def merge_and_clean_data(data_frames):
    """
    合并数据并清洗
    
    参数:
        data_frames: {trade_code: DataFrame} 字典
        
    返回:
        pd.DataFrame: 合并后的数据
    """
    # 合并三个数据集
    merged_df = pd.concat([
        data_frames['CRB CMDT Index']['CLOSE'].rename('CRB'),
        data_frames['FLOT US Equity']['CLOSE'].rename('FLOT'),
        data_frames['.EARNREV G Index']['CLOSE'].rename('EARNREV')
    ], axis=1)
    
    # 对缺失数据进行插值
    merged_df = merged_df.interpolate(method='linear')
    
    # 删除仍有缺失值的行
    merged_df = merged_df.dropna()
    
    logger.info(f"合并后数据共 {len(merged_df)} 条记录")
    
    return merged_df

# 合并并清洗数据
merged_df = merge_and_clean_data(data_frames)
merged_df.head(10)

### 4.2 数据转换

In [ ]:
def calculate_ratio(merged_df, base_date, target_base_value):
    """
    计算比值并按基准调整
    
    参数:
        merged_df: 合并后的数据
        base_date: 基准日期
        target_base_value: 目标基准值
        
    返回:
        pd.DataFrame: 处理后的数据
    """
    # 计算比值：CRB / (0.5 * (FLOT + EARNREV))
    merged_df['ratio'] = merged_df['CRB'] / (0.5 * (merged_df['FLOT'] + merged_df['EARNREV']))
    
    # 基准调整
    base_date_ts = pd.to_datetime(base_date)
    if base_date_ts in merged_df.index:
        base_value = merged_df.loc[base_date_ts, 'ratio']
        adjustment_factor = target_base_value / base_value
        merged_df['close'] = merged_df['ratio'] * adjustment_factor
        logger.info(f"基准调整因子: {adjustment_factor:.6f}")
    else:
        logger.warning(f"未找到基准日期 {base_date} 的数据，使用原始比值")
        merged_df['close'] = merged_df['ratio']
    
    return merged_df

def convert_to_weekly(data_df):
    """
    转换为周度数据，取每周最后一个自然日
    
    参数:
        data_df: 日度数据DataFrame
        
    返回:
        pd.DataFrame: 周度数据DataFrame
    """
    # 创建结果DataFrame
    result_df = pd.DataFrame({
        'dt': data_df.index,
        'close': data_df['close']
    })
    
    result_df['dt'] = pd.to_datetime(result_df['dt'])
    result_df = result_df.set_index('dt')
    
    # 按周重采样，取每周最后一个值
    weekly_df = result_df.resample('W').last()
    
    # 重置索引
    weekly_df = weekly_df.reset_index()
    weekly_df.columns = ['dt', 'close']
    
    # 删除缺失值
    weekly_df = weekly_df.dropna()
    
    logger.info(f"周度数据共 {len(weekly_df)} 条记录")
    
    return weekly_df

# 计算比率
ratio_df = calculate_ratio(merged_df, BASE_DATE, TARGET_BASE_VALUE)

# 转换为周度数据
weekly_df = convert_to_weekly(ratio_df)
weekly_df.head(10)

## 5. 核心逻辑

### 5.1 商品比率计算

In [ ]:
def insert_to_database(data_df, engine, table_name, base_date):
    """
    将数据插入到数据库表中
    
    参数:
        data_df: 要插入的数据DataFrame
        engine: SQLAlchemy引擎
        table_name: 目标表名
        base_date: 基准日期（该日期的数据将被过滤）
    """
    logger.info(f"开始插入数据到数据库表 {table_name}...")
    
    # 过滤掉基准日期的数据
    base_date_ts = pd.to_datetime(base_date)
    data_df = data_df[data_df['dt'] > base_date_ts]
    
    if len(data_df) == 0:
        logger.warning("没有需要插入的数据")
        return 0
    
    logger.info(f"需要插入 {len(data_df)} 条记录")
    
    try:
        # 使用to_sql插入数据
        data_df.to_sql(
            table_name,
            engine,
            if_exists='append',
            index=False,
            chunksize=1000
        )
        
        logger.info(f"成功插入 {len(data_df)} 条记录")
        return len(data_df)
        
    except Exception as e:
        logger.error(f"数据插入失败: {e}")
        raise

# 插入数据（取消注释以执行）
# rows_inserted = insert_to_database(weekly_df, engine, '全球实物资产比金融资产', BASE_DATE)
print("数据插入函数已定义，取消上方注释以执行插入操作")

### 5.2 图表数据提取

In [ ]:
def select_data_region(img):
    """交互式选择数据区域"""
    window_name = '选择数据区域 - 按Enter确认，按c重选'
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    
    # 调整窗口大小以适应屏幕
    screen_height = 800
    screen_width = int(img.shape[1] * screen_height / img.shape[0])
    cv2.resizeWindow(window_name, min(screen_width, 1200), screen_height)
    
    while True:
        img_copy = img.copy()
        roi = cv2.selectROI(window_name, img_copy, False, False)
        
        x, y, w, h = roi
        cv2.rectangle(img_copy, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.imshow(window_name, img_copy)
        
        key = cv2.waitKey(0) & 0xFF
        if key == 13:  # Enter键
            cv2.destroyWindow(window_name)
            return roi
        elif key == ord('c'):  # c键重选
            continue
        else:
            cv2.destroyWindow(window_name)
            return roi

In [ ]:
def get_axis_range():
    """
    获取坐标轴范围（交互式输入）
    在实际使用时需要用户手动输入
    """
    # 示例值，实际使用时替换为交互式输入
    print("请输入Y轴范围（数值）：")
    print("请输入X轴范围（日期，格式：YYYY/MM 或 YYYY-MM-DD）：")
    
    # 默认示例值
    y_min, y_max = 0.05, 0.15
    start_date = datetime(2020, 11, 1)
    end_date = datetime(2025, 1, 1)
    
    return (y_min, y_max), (start_date, end_date)

In [ ]:
def select_sample_point(img, cache_file="sample_points_cache.json"):
    """
    交互式选择曲线的样本点
    
    参数:
        img: 图像
        cache_file: 缓存文件路径
        
    返回:
        list: 样本点列表 [(x, y), ...]
    """
    window_name = '选择曲线样本点 - 左键选择点，ESC清除所有点，Enter确认完成'
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    
    screen_height = 800
    screen_width = int(img.shape[1] * screen_height / img.shape[0])
    cv2.resizeWindow(window_name, min(screen_width, 1200), screen_height)
    
    # 检查是否存在缓存文件
    if os.path.exists(cache_file):
        print(f"检测到已保存的样本点缓存: {cache_file}")
        try:
            with open(cache_file, 'r', encoding='utf-8') as f:
                cached_points = json.load(f)
            print(f"已加载 {len(cached_points)} 个缓存点")
            return [(p[0], p[1]) for p in cached_points]
        except Exception as e:
            print(f"加载缓存失败: {e}")
    
    img_copy = img.copy()
    points_data = {'points': []}
    
    def mouse_callback(event, x, y, flags, param):
        nonlocal img_copy
        if event == cv2.EVENT_LBUTTONDOWN:
            param['points'].append((x, y))
            temp_img = img.copy()
            for px, py in param['points']:
                cv2.circle(temp_img, (px, py), 3, (0, 255, 0), -1)
            cv2.imshow(window_name, temp_img)
            img_copy = temp_img
    
    cv2.setMouseCallback(window_name, mouse_callback, points_data)
    cv2.imshow(window_name, img_copy)
    
    while True:
        key = cv2.waitKey(1) & 0xFF
        if key == 13 and points_data['points']:  # Enter键
            try:
                with open(cache_file, 'w', encoding='utf-8') as f:
                    json.dump(points_data['points'], f)
                print(f"已保存 {len(points_data['points'])} 个样本点到缓存文件")
            except Exception as e:
                print(f"保存缓存失败: {e}")
            
            cv2.destroyWindow(window_name)
            return points_data['points']
        elif key == 27:  # ESC键清除所有点
            img_copy = img.copy()
            cv2.imshow(window_name, img_copy)
            points_data['points'] = []

In [ ]:
def extract_data_from_graph(image_path, is_url=False, frequency='week'):
    """
    从图表图像中提取数据点
    
    参数:
        image_path: 图像路径或URL
        is_url: 是否为URL
        frequency: 数据点频率 ('week', 'month', 'year')
        
    返回:
        value_points: 数据点列表，每个元素为(日期, 值)元组
        result_img: 标注了检测点的结果图像
    """
    # 读取图片
    if is_url:
        response = requests.get(image_path)
        img_array = np.asarray(bytearray(response.content), dtype=np.uint8)
    else:
        with open(image_path, 'rb') as f:
            img_array = np.asarray(bytearray(f.read()), dtype=np.uint8)
    
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    
    print("请框选数据点区域（包含所有数据点的矩形区域）")
    data_roi = select_data_region(img)
    x, y, w, h = data_roi
    
    # 获取坐标轴范围
    (y_min, y_max), (start_date, end_date) = get_axis_range()
    
    # 提取数据点区域
    points_region = img[y:y+h, x:x+w]
    
    # 交互式选择曲线的样本点
    print("请选择曲线上的多个样本点，按Enter结束选择")
    sample_points = select_sample_point(img)
    
    # 获取所有样本点的颜色，计算颜色范围
    sample_colors = []
    for point in sample_points:
        color = img[point[1], point[0]]
        sample_colors.append(color)
    
    if len(sample_colors) > 0:
        sample_colors = np.array(sample_colors)
        avg_sample_color = np.mean(sample_colors, axis=0)
        color_std = np.std(sample_colors, axis=0)
        tolerance = np.maximum(color_std * 2, [10, 10, 10])
        print(f"\n样本点平均颜色: {avg_sample_color}")
        print(f"颜色容差: {tolerance}")
    else:
        avg_sample_color = np.array([0, 0, 0])
        tolerance = np.array([25, 25, 25])
    
    # 创建颜色掩码
    mask = np.zeros(points_region.shape[:2], dtype=np.uint8)
    
    for i in range(points_region.shape[0]):
        for j in range(points_region.shape[1]):
            pixel_color = points_region[i, j]
            color_diff = np.abs(pixel_color.astype(np.float32) - avg_sample_color)
            if np.all(color_diff <= tolerance):
                mask[i, j] = 255
    
    # 形态学操作清理噪点
    kernel_open = np.ones((2, 2), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    
    kernel_close = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)
    
    kernel_line = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 1))
    mask = cv2.dilate(mask, kernel_line, iterations=1)
    mask = cv2.erode(mask, kernel_close, iterations=1)
    
    # 查找轮廓
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    print(f"\n找到的轮廓数量: {len(contours)}")
    
    # 收集所有点
    points = []
    for contour in contours:
        for point in contour:
            px = point[0][0] + x
            py = point[0][1] + y
            points.append((px, py))
    
    # 去重并排序
    points = list(set(points))
    points.sort(key=lambda p: p[0])
    
    # 根据频率采样
    if len(points) > 0:
        months_range = (end_date.year - start_date.year) * 12 + end_date.month - start_date.month
        weeks_range = int(months_range * 4.33)
        target_points = max(30, min(weeks_range, len(points)))
        
        if len(points) > target_points and target_points > 2:
            sampled_points = [points[0]]
            step = (len(points) - 2) / (target_points - 2)
            for i in range(1, target_points - 1):
                idx = int(i * step) + 1
                if idx < len(points) - 1:
                    sampled_points.append(points[idx])
            sampled_points.append(points[-1])
            points = sampled_points
    
    # 转换坐标为实际值
    def pixel_to_value(point):
        px, py = point
        
        y_value = y_max - (py - y) * (y_max - y_min) / h
        days_range = (end_date - start_date).days
        days_from_start = (px - x) * days_range / w
        date = start_date + timedelta(days=days_from_start)
        
        return (date.strftime('%Y/%m/%d'), y_value)
    
    value_points = [pixel_to_value(p) for p in points]
    
    # 日期去重
    unique_value_points = {}
    for date_str, value in value_points:
        if date_str not in unique_value_points:
            unique_value_points[date_str] = value
    
    value_points = list(unique_value_points.items())
    value_points.sort(key=lambda x: x[0])
    
    # 可视化结果
    result_img = img.copy()
    cv2.rectangle(result_img, (x, y), (x+w, y+h), (0, 255, 0), 2)
    
    for px, py in points:
        cv2.circle(result_img, (px, py), 2, (0, 0, 255), -1)
    
    for px, py in sample_points:
        cv2.circle(result_img, (px, py), 4, (0, 255, 0), -1)
    
    return value_points, result_img

print("图表数据提取函数已定义")

### 5.3 辅助函数

In [ ]:
import re

def clean_province_name(name):
    """
    清理省级名称
    
    参数:
        name: 省份名称
        
    返回:
        str: 清理后的名称
    """
    if pd.isna(name):
        return name
    return re.sub(r'省|市|自治区|回族自治区|维吾尔自治区|壮族自治区', '', name)

def parse_date(date_str):
    """
    解析日期字符串，支持多种格式
    
    参数:
        date_str: 日期字符串
        
    返回:
        datetime: 解析后的日期对象
    """
    formats = ['%Y/%m', '%Y-%m-%d', '%Y/%m/%d', '%Y-%m']
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    raise ValueError(f"无法解析日期格式: {date_str}")

print("辅助函数已定义")

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
def run_commodity_ratio_workflow():
    """
    执行商品比率计算工作流
    """
    try:
        logger.info("开始执行商品指数比率计算任务...")
        
        # 1. 查询数据
        data_frames = get_data_from_database(engine, TRADE_CODES, BASE_DATE)
        
        # 2. 合并并清洗数据
        merged_df = merge_and_clean_data(data_frames)
        
        # 3. 计算比率
        ratio_df = calculate_ratio(merged_df, BASE_DATE, TARGET_BASE_VALUE)
        
        # 4. 转换为周度数据
        weekly_data = convert_to_weekly(ratio_df)
        
        # 5. 插入数据库（取消注释以执行）
        # insert_to_database(weekly_data, engine, '全球实物资产比金融资产', BASE_DATE)
        
        logger.info("任务执行完成")
        return weekly_data
        
    except Exception as e:
        logger.error(f"任务执行失败: {e}")
        raise

# 执行工作流
weekly_result = run_commodity_ratio_workflow()
print(f"\n结果数据共 {len(weekly_result)} 条记录")
weekly_result.tail(10)

### 6.2 结果验证

In [ ]:
def verify_results(engine, table_name='全球实物资产比金融资产'):
    """
    验证数据插入结果
    
    参数:
        engine: SQLAlchemy引擎
        table_name: 表名
    """
    try:
        # 查询表中数据总数
        count_sql = f"SELECT COUNT(*) as count FROM `{table_name}`"
        count_df = pd.read_sql(count_sql, engine)
        print(f'表中数据总数: {count_df["count"].iloc[0]}')
        
        # 查询最新5条数据
        latest_sql = f"SELECT dt, close FROM `{table_name}` ORDER BY dt DESC LIMIT 5"
        latest_df = pd.read_sql(latest_sql, engine)
        print('\n最新5条数据:')
        print(latest_df)
        
        # 查询最早的5条数据
        earliest_sql = f"SELECT dt, close FROM `{table_name}` ORDER BY dt ASC LIMIT 5"
        earliest_df = pd.read_sql(earliest_sql, engine)
        print('\n最早5条数据:')
        print(earliest_df)
        
    except Exception as e:
        print(f"验证失败: {e}")

# 执行验证（取消注释以执行）
# verify_results(engine)
print("验证函数已定义，取消上方注释以执行验证操作")

In [ ]:
# 可视化结果数据
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 绘制结果图表
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(weekly_result['dt'], weekly_result['close'], 'b-', linewidth=1.5, label='全球实物资产比金融资产')
ax.set_xlabel('日期')
ax.set_ylabel('比率')
ax.set_title('全球实物资产比金融资产 - 周度数据')
ax.legend()
ax.grid(True, alpha=0.3)

# 格式化x轴日期
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('output/全球实物资产比金融资产.png', dpi=150, bbox_inches='tight')
plt.show()

print("图表已保存到 output/全球实物资产比金融资产.png")

## 7. 工具函数（可复用）

In [ ]:
def save_dataframe_to_database(df, engine, table_name, if_exists='append'):
    """
    将DataFrame保存到数据库
    
    参数:
        df: 要保存的DataFrame
        engine: SQLAlchemy引擎
        table_name: 目标表名
        if_exists: 如果表存在的处理方式 ('fail', 'replace', 'append')
    """
    df.to_sql(table_name, engine, if_exists=if_exists, index=False)
    logger.info(f"成功保存 {len(df)} 条记录到表 {table_name}")

def load_dataframe_from_database(engine, sql, params=None):
    """
    从数据库加载数据到DataFrame
    
    参数:
        engine: SQLAlchemy引擎
        sql: SQL查询语句
        params: 查询参数
        
    返回:
        pd.DataFrame: 查询结果
    """
    return pd.read_sql(sql, engine, params=params)

def save_dataframe_to_csv(df, filepath):
    """
    将DataFrame保存到CSV文件
    
    参数:
        df: 要保存的DataFrame
        filepath: 目标文件路径
    """
    df.to_csv(filepath, index=False, encoding='utf-8-sig')
    logger.info(f"成功保存数据到 {filepath}")

print("工具函数已定义")

In [ ]:
# 保存结果到CSV
output_csv_path = 'output/全球实物资产比金融资产_周度数据.csv'
save_dataframe_to_csv(weekly_result, output_csv_path)
print(f"结果已保存到: {output_csv_path}")